# HAGI - Update HF Model Cards

Run this **after** the geo-diagnostic. Pushes a fresh, cross-linked README to each
ablation repo (`hagi-ablation-a/b/c/d`): TL;DR + headline result, full architecture,
results, seed-stability, a **clickable Model-family nav** (Stage 0 + A/B/C/D), and -
if you paste the geo verdict below - a **Geometry-diagnostic** section.

**Does NOT touch `hagi-stage0`** (only links to it), so your existing Stage 0 card
is left intact.

## Before running
1. **Add-ons -> Secrets -> add `HF_TOKEN`** (a HF token with **write** access). Cell 3
   reads it; the token is never printed or committed.
2. **Settings -> Internet -> On**. No GPU needed (CPU is fine - this only uploads text).

## 1. Clone + update the repo

In [ ]:
import os
%cd /kaggle/working
if not os.path.isdir('HAGI'):
    !git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /kaggle/working/HAGI
!git pull --ff-only origin experimental
print('cwd:', os.getcwd())

## 2. Install the Hub client

In [ ]:
!pip install -q -U huggingface_hub

## 3. Knobs + login (HF_TOKEN from Kaggle secret)

In [ ]:
# ---- knobs ----
USER   = 'NAME0x0'           # your HF username (repos: <USER>/hagi-ablation-{a,b,c,d})
MODELS = 'a,b,c,d'           # which cards to push
# Paste the geo verdict here AFTER the diagnostic run, else leave GEO = None.
# From the GEO_DIAG line: mean_D-B -> dnb, mean_Dnogeo-B -> dngb.
GEO    = None                # e.g. {'seeds': [1, 2], 'dnb': 0.0180, 'dngb': 0.0050}
# ---------------
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secret')
except Exception as e:
    assert os.environ.get('HF_TOKEN'), f'No HF_TOKEN. Add-ons -> Secrets -> HF_TOKEN (write). ({e})'
    print('using HF_TOKEN from environment')
from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
print('logged in as:', whoami()['name'])

## 4. Preview one card (no upload) - sanity check before pushing

In [ ]:
import json
geo_arg = []
if GEO:
    json.dump(GEO, open('/kaggle/working/geo.json', 'w'))
    geo_arg = ['--geo-json', '/kaggle/working/geo.json']
    print('geo verdict will be included:', GEO)
else:
    print('GEO is None -> cards push without the Geometry-diagnostic section')
!python scripts/push_model_cards.py --user {USER} --models {MODELS.split(',')[-1]} {' '.join(geo_arg)} --dry-run | head -40

## 5. Push the cards

In [ ]:
import subprocess
subprocess.run(['python', 'scripts/push_model_cards.py', '--user', USER,
                '--models', MODELS, *geo_arg], check=True)
print('\ndone - open the repos:')
for m in MODELS.split(','):
    print(f'  https://huggingface.co/{USER}/hagi-ablation-{m.strip()}')